# Is it useful ?!

In [1]:
import os
import pandas as pd

DATA_DIR = "/nfs/scratch/pdb_dimers/KaHIP"

In [2]:
output = pd.read_csv(os.path.join(DATA_DIR, "output.txt"), header=None)
output.head()

,0
0,3
1,3
2,3
3,3
4,3


In [3]:
output["cluster_id"] = output.index.map(lambda x: x + 1)  # METIS uses 1-based indexing
output.head()

,0,cluster_id
0,3,1
1,3,2
2,3,3
3,3,4
4,3,5


In [4]:
# Plot the distribution of cluster sizes
cluster_sizes = output[0].value_counts().sort_index()
print(cluster_sizes)

0
0    799
1    799
2    798
3    796
4    798
5    799
6    798
7    798
8    799
9    799
Name: count, dtype: int64


# Partitioning with sequence identity

In [5]:
seq_ident_output = pd.read_csv(os.path.join(DATA_DIR, "seq_ident_partitions_strong_output.txt"), header=None)
seq_ident_output.head()

,0
0,0
1,7
2,0
3,7
4,2


In [6]:
len(seq_ident_output)

38758

In [7]:
seq_ident_cluster_sizes = seq_ident_output[0].value_counts().sort_index()
print(seq_ident_cluster_sizes)

0
0    3991
1    3991
2    3991
3    2839
4    3991
5    3991
6    3991
7    3991
8    3991
9    3991
Name: count, dtype: int64


# Fuse partition plus into interaction df

In [8]:
int_df = pd.read_csv("/nfs/scratch/pdb_dimers/dataset_iterations/04_removed_duplicates.tsv", sep="\t")
partitions = pd.read_csv("/nfs/scratch/pdb_dimers/KaHIP/seq_ident_partitions_strong_output.txt", header=None)

partitions.head()

,0
0,0
1,7
2,0
3,7
4,2


In [9]:
len(partitions)

38758

In [10]:
partitions = partitions[0]

int_df["partitions"] = partitions

int_df.head()

,assembly_id,pdb_id,assembly_number,entity_pair,uniprot_pair,uniprot_1,uniprot_2,species_pair,species_1,species_2,...,dimer_type,cluster_pair_100pct,new_cluster_pair,resolution_best_angstrom,modeled_polymer_monomer_count,experimental_method,oligomeric_count,download_url,local_filename,partitions
0,10BL-1,10BL,1,"10BL_1,10BL_1",Q4E2L0|Q4E2L0,Q4E2L0,Q4E2L0,Trypanosoma cruzi|Trypanosoma cruzi,Trypanosoma cruzi,Trypanosoma cruzi,...,homo,"21081_100,21081_100","fix_19099_100,fix_19099_100",2.60,664,X-ray,2,https://files.rcsb.org/download/10bl-assembly1...,10bl-assembly1.cif.gz,0
1,10BR-1,10BR,1,"10BR_1,10BR_1",O51131|O51131,O51131,O51131,Borreliella burgdorferi B31|Borreliella burgdo...,Borreliella burgdorferi B31,Borreliella burgdorferi B31,...,homo,"56122_100,56122_100","fix_8775_100,fix_8775_100",1.50,403,X-ray,2,https://files.rcsb.org/download/10br-assembly1...,10br-assembly1.cif.gz,7
2,10BT-1,10BT,1,"10BT_1,10BT_2",|,NaN,NaN,Homo sapiens|Homo sapiens,Homo sapiens,Homo sapiens,...,hetero,"156562_100,156563_100","fix_4330_100,fix_2529_100",1.99,438,X-ray,2,https://files.rcsb.org/download/10bt-assembly1...,10bt-assembly1.cif.gz,0
3,10BV-1,10BV,1,"10BV_1,10BV_1",A0A060IEI9|A0A060IEI9,A0A060IEI9,A0A060IEI9,Lactobacillus helveticus|Lactobacillus helveticus,Lactobacillus helveticus,Lactobacillus helveticus,...,homo,"157883_100,157883_100","fix_38480_100,fix_38480_100",3.05,496,X-ray,2,https://files.rcsb.org/download/10bv-assembly1...,10bv-assembly1.cif.gz,7
4,10DV-1,10DV,1,"10DV_1,10DV_1",P0DTD1;P19909|P0DTD1;P19909,P0DTD1;P19909,P0DTD1;P19909,Homo sapiens|Homo sapiens,Homo sapiens,Homo sapiens,...,homo,"86315_100,86315_100","fix_37354_100,fix_37354_100",2.05,602,X-ray,2,https://files.rcsb.org/download/10dv-assembly1...,10dv-assembly1.cif.gz,2


In [11]:
all_homo = 0
all_hetero = 0
for i in range(10):
    int_df_tmp = int_df[int_df["partitions"] == i]
    counts = int_df_tmp["dimer_type"].value_counts(normalize=True)*100
    all_homo += counts["homo"]
    all_hetero += counts["hetero"]
    print(f"{i}:\thomo: {counts["homo"]}\thetero: {counts["hetero"]}")

print("\n")
print(f"Avg Counts:\nhomo: {all_homo/10}\thetero: {all_hetero/10}")

0:	homo: 32.548233525432224	hetero: 67.45176647456778
1:	homo: 60.53620646454523	hetero: 39.46379353545477
2:	homo: 83.5630167877725	hetero: 16.436983212227513
3:	homo: 83.1983092638253	hetero: 16.80169073617471
4:	homo: 87.27136056126284	hetero: 12.728639438737158
5:	homo: 79.57905286895515	hetero: 20.42094713104485
6:	homo: 89.7018291155099	hetero: 10.298170884490103
7:	homo: 93.63568028063142	hetero: 6.364319719368579
8:	homo: 90.45352042094713	hetero: 9.54647957905287
9:	homo: 89.02530694061639	hetero: 10.974693059383613


Avg Counts:
homo: 78.9512516229498	hetero: 21.048748377050195


In [12]:
int_df["split"] = int_df["partitions"].map({
    0: "train",
    1: "train",
    2: "train",
    3: "test",
    4: "train",
    5: "val",
    6: "train",
    7: "train",
    8: "train",
    9: "train",
})

In [13]:
int_df.to_csv("/nfs/scratch/pdb_dimers/dataset_iterations/05_KaHIP_partitions.tsv", sep="\t", index=False)

# homo vs hetero distribution in different splits

In [14]:
int_df = pd.read_csv("/nfs/scratch/pdb_dimers/dataset_iterations/05_KaHIP_partitions.tsv", sep="\t")


int_df_train = int_df[int_df["split"]=="train"]
int_df_val = int_df[int_df["split"]=="val"]
int_df_test = int_df[int_df["split"]=="test"]

len(int_df_test)

2839

In [ ]:
def calculate_percentages(df):
    return df["dimer_type"].value_counts(normalize=True) * 100


calculate_percentages(int_df_train)

dimer_type
homo      78.341894
hetero    21.658106
Name: proportion, dtype: float64

In [16]:
calculate_percentages(int_df_val)

dimer_type
homo      79.579053
hetero    20.420947
Name: proportion, dtype: float64

In [17]:
calculate_percentages(int_df_test)

dimer_type
homo      83.198309
hetero    16.801691
Name: proportion, dtype: float64